# Eval Metric and Runtime Deltas

This notebook is intentionally split into stages:

1. Extract eval metrics from `eval_results/`
2. Inspect the parsed eval table locally
3. Query W&B Local for matching training runs and runtimes
4. Join both sources
5. Compute baseline-relative metric deltas and runtime percent deltas using `fixed_k_max`


In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import wandb


In [2]:
def resolve_repo_root():
    candidates = [Path.cwd(), Path.cwd().resolve(), Path(".").resolve(), Path("..").resolve()]

    try:
        candidates.extend(Path(__file__).resolve().parents)
    except NameError:
        pass

    notebook_hint = Path("notebooks/eval_metric_runtime_deltas.ipynb").resolve()
    candidates.extend(notebook_hint.parents)

    for candidate in candidates:
        if (candidate / "eval_results").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError(
        f"Could not locate repo root from cwd={Path.cwd()}"
    )


ROOT = resolve_repo_root()
EVAL_ROOT = ROOT / "eval_results"
OUTPUT_CSV = ROOT / "notebooks" / "eval_metric_runtime_deltas.csv"

WANDB_BASE_URL = os.environ.get("WANDB_BASE_URL", "http://localhost:8080")
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "sebi")
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "routing_curriculum")

METRIC_KEYS = {
    "arc": "accuracy",
    "sciq": "accuracy",
    "gsm8k": "accuracy_extracted_answer",
}

EVAL_VARIANT_PRIORITY = {
    "best": 0,
    "final": 1,
    "default": 2,
}

CURRICULUM_DISPLAY_ORDER = [
    "fixed_k_max",
    "fixed_k_1",
    "linear_k_1_to_topk",
    "frontloaded",
    "backloaded",
    "warmup",
    "jump_warmup",
    "linear_mid_start"
]

ROOT, EVAL_ROOT, OUTPUT_CSV

(PosixPath('/home/sebi/disertatie'),
 PosixPath('/home/sebi/disertatie/eval_results'),
 PosixPath('/home/sebi/disertatie/notebooks/eval_metric_runtime_deltas.csv'))

## 1. Extract Eval Results Only

This section only reads local files under `eval_results/`. No W&B calls happen yet.


In [3]:
def metric_key_for(metrics):
    benchmark = metrics["benchmark"]
    metric_key = METRIC_KEYS.get(benchmark)
    if metric_key is None or metric_key not in metrics:
        raise KeyError(
            f"Unsupported benchmark metric for {benchmark}: {sorted(metrics.keys())}"
        )
    return metric_key


def parse_run_name(run_name, benchmark):
    if "__seed_" not in run_name:
        return None

    seeded_run_name, seed = run_name.split("__seed_", 1)
    try:
        seed = int(seed)
    except ValueError:
        pass

    train_run_name = run_name
    normalized_run_name = seeded_run_name
    split_token = f"_{benchmark}_"
    if split_token not in normalized_run_name:
        return None

    model_family, suffix = normalized_run_name.split(split_token, 1)
    curriculum = suffix[:-5] if suffix.endswith("_lora") else suffix
    return model_family, curriculum, train_run_name, seed


EVAL_COLUMNS = [
    "benchmark",
    "model_family",
    "curriculum",
    "run_name",
    "train_run_name",
    "seed",
    "metric_name",
    "metric_value",
    "num_examples",
    "checkpoint_path",
    "model_id",
    "metrics_path",
]


def load_eval_table():
    rows = []

    for metrics_path in sorted(EVAL_ROOT.glob("*/*/metrics.json")):
        metrics = json.loads(metrics_path.read_text())
        run_name = metrics_path.parent.name
        benchmark = metrics["benchmark"]
        parsed = parse_run_name(run_name, benchmark)

        if parsed is None:
            continue

        model_family, curriculum, train_run_name, seed = parsed
        metric_key = metric_key_for(metrics)

        rows.append(
            {
                "benchmark": benchmark,
                "model_family": model_family,
                "curriculum": curriculum,
                "run_name": run_name,
                "train_run_name": train_run_name,
                "seed": seed,
                "metric_name": metric_key,
                "metric_value": metrics[metric_key],
                "num_examples": metrics.get("num_examples"),
                "checkpoint_path": metrics.get("checkpoint_path"),
                "model_id": metrics.get("model_id"),
                "metrics_path": str(metrics_path),
            }
        )

    eval_df = pd.DataFrame(rows, columns=EVAL_COLUMNS)
    if eval_df.empty:
        return eval_df

    return eval_df.sort_values(
        ["benchmark", "model_family", "curriculum", "seed", "run_name"]
    ).reset_index(drop=True)


In [4]:
eval_df = load_eval_table()
print(f"Eval rows: {len(eval_df)}")
eval_df

Eval rows: 288


,benchmark,model_family,curriculum,run_name,train_run_name,seed,metric_name,metric_value,num_examples,checkpoint_path,model_id,metrics_path
0,arc,gpt_oss,backloaded,gpt_oss_arc_backloaded_lora__seed_42,gpt_oss_arc_backloaded_lora__seed_42,42,accuracy,0.876280,1172,ckpt/gpt_oss_arc_backloaded_lora__seed_42/fina...,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
1,arc,gpt_oss,backloaded,gpt_oss_arc_backloaded_lora__seed_123,gpt_oss_arc_backloaded_lora__seed_123,123,accuracy,0.869454,1172,ckpt/gpt_oss_arc_backloaded_lora__seed_123/fin...,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
2,arc,gpt_oss,backloaded,gpt_oss_arc_backloaded_lora__seed_999,gpt_oss_arc_backloaded_lora__seed_999,999,accuracy,0.865188,1172,ckpt/gpt_oss_arc_backloaded_lora__seed_999/fin...,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
3,arc,gpt_oss,fixed_k_1,gpt_oss_arc_fixed_k_1_lora__seed_42,gpt_oss_arc_fixed_k_1_lora__seed_42,42,accuracy,0.747440,1172,ckpt/gpt_oss_arc_fixed_k_1_lora__seed_42/final...,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
4,arc,gpt_oss,fixed_k_1,gpt_oss_arc_fixed_k_1_lora__seed_123,gpt_oss_arc_fixed_k_1_lora__seed_123,123,accuracy,0.763652,1172,ckpt/gpt_oss_arc_fixed_k_1_lora__seed_123/fina...,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
...,...,...,...,...,...,...,...,...,...,...,...,...
283,sciq,qwen,linear_mid_start,qwen_sciq_linear_mid_start_lora__seed_123,qwen_sciq_linear_mid_start_lora__seed_123,123,accuracy,0.927000,1000,ckpt/qwen_sciq_linear_mid_start_lora__seed_123...,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...
284,sciq,qwen,linear_mid_start,qwen_sciq_linear_mid_start_lora__seed_999,qwen_sciq_linear_mid_start_lora__seed_999,999,accuracy,0.923000,1000,ckpt/qwen_sciq_linear_mid_start_lora__seed_999...,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...
285,sciq,qwen,warmup,qwen_sciq_warmup_lora__seed_42,qwen_sciq_warmup_lora__seed_42,42,accuracy,0.922000,1000,ckpt/qwen_sciq_warmup_lora__seed_42/final_adapter,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...
286,sciq,qwen,warmup,qwen_sciq_warmup_lora__seed_123,qwen_sciq_warmup_lora__seed_123,123,accuracy,0.929000,1000,ckpt/qwen_sciq_warmup_lora__seed_123/final_ada...,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...


In [5]:
eval_df.groupby(["benchmark", "model_family"])["curriculum"].agg(list)

benchmark  model_family
arc        gpt_oss         [backloaded, backloaded, backloaded, fixed_k_1...
           lfm2            [backloaded, backloaded, backloaded, fixed_k_1...
           olmoe           [backloaded, backloaded, backloaded, fixed_k_1...
           qwen            [backloaded, backloaded, backloaded, fixed_k_1...
gsm8k      gpt_oss         [backloaded, backloaded, backloaded, fixed_k_1...
           lfm2            [backloaded, backloaded, backloaded, fixed_k_1...
           olmoe           [backloaded, backloaded, backloaded, fixed_k_1...
           qwen            [backloaded, backloaded, backloaded, fixed_k_1...
sciq       gpt_oss         [backloaded, backloaded, backloaded, fixed_k_1...
           lfm2            [backloaded, backloaded, backloaded, fixed_k_1...
           olmoe           [backloaded, backloaded, backloaded, fixed_k_1...
           qwen            [backloaded, backloaded, backloaded, fixed_k_1...
Name: curriculum, dtype: object

## 2. Query W&B Local

This section queries W&B only after the eval table is already extracted and visible.

Set these first if needed:

- `WANDB_BASE_URL`
- `WANDB_API_KEY`
- `WANDB_ENTITY`
- `WANDB_PROJECT`


In [6]:
if WANDB_ENTITY == "REPLACE_ME":
    raise ValueError(
        "Set WANDB_ENTITY before running the W&B cells. "
        "Example: export WANDB_ENTITY=your-team-or-username"
    )

api = wandb.Api(overrides={"base_url": WANDB_BASE_URL}, timeout=60)
WANDB_BASE_URL, WANDB_ENTITY, WANDB_PROJECT

wandb: Loading settings from /home/sebi/.config/wandb/settings
wandb: [wandb.Api()] Loaded credentials for http://localhost:8080 from /home/sebi/.netrc.


('http://localhost:8080', 'sebi', 'routing_curriculum')

In [7]:
def load_wandb_runtime_table(eval_table):
    rows = []

    eval_name_lookup = {}
    eval_run_pairs = eval_table[["run_name", "train_run_name"]].drop_duplicates()
    for row in eval_run_pairs.itertuples(index=False):
        eval_name_lookup[row.run_name] = row.run_name
        eval_name_lookup[row.train_run_name] = row.run_name

    runs = api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}")

    for run in runs:
        config = dict(run.config or {})
        summary = dict(run.summary or {})

        output_dir = config.get("output_dir")
        output_run_name = Path(output_dir).name if output_dir else None
        candidate_run_names = [value for value in [output_run_name, run.name] if value]

        matched_eval_run_name = next(
            (
                eval_name_lookup.get(candidate_run_name)
                for candidate_run_name in candidate_run_names
                if eval_name_lookup.get(candidate_run_name) is not None
            ),
            None,
        )
        if matched_eval_run_name is None:
            continue

        training_runtime_seconds = summary.get("training_runtime_seconds")

        matched_train_run_name = next(
            (
                candidate_run_name
                for candidate_run_name in candidate_run_names
                if eval_name_lookup.get(candidate_run_name) == matched_eval_run_name
            ),
            None,
        )

        rows.append(
            {
                "run_name": matched_eval_run_name,
                "train_run_name": matched_train_run_name,
                "wandb_run_name": run.name,
                "wandb_run_id": run.id,
                "wandb_run_path": "/".join(run.path),
                "wandb_run_url": getattr(run, "url", None),
                "created_at": getattr(run, "created_at", None),
                "state": getattr(run, "state", None),
                "wandb_project": WANDB_PROJECT,
                "dataset_id": config.get("dataset_id"),
                "output_dir": output_dir,
                "training_runtime_seconds": training_runtime_seconds,
            }
        )

    expected_columns = [
        "run_name",
        "train_run_name",
        "wandb_run_name",
        "wandb_run_id",
        "wandb_run_path",
        "wandb_run_url",
        "created_at",
        "state",
        "wandb_project",
        "dataset_id",
        "output_dir",
        "training_runtime_seconds",
    ]

    wandb_df = pd.DataFrame(rows, columns=expected_columns)
    if wandb_df.empty:
        return wandb_df, pd.DataFrame(columns=["run_name", "num_wandb_matches"])

    duplicate_counts = (
        wandb_df.groupby("run_name")
        .size()
        .rename("num_wandb_matches")
        .reset_index()
        .query("num_wandb_matches > 1")
        .sort_values(["num_wandb_matches", "run_name"], ascending=[False, True])
        .reset_index(drop=True)
    )

    wandb_df = wandb_df.sort_values(["created_at", "wandb_run_id"]).drop_duplicates(
        subset="run_name", keep="last"
    )

    return wandb_df.reset_index(drop=True), duplicate_counts


In [8]:
wandb_df, duplicate_wandb_runs = load_wandb_runtime_table(eval_df)
print(f"Matched W&B rows: {len(wandb_df)}")
wandb_df.sort_values(["run_name"]).reset_index(drop=True)


Matched W&B rows: 288


,run_name,train_run_name,wandb_run_name,wandb_run_id,wandb_run_path,wandb_run_url,created_at,state,wandb_project,dataset_id,output_dir,training_runtime_seconds
0,gpt_oss_arc_backloaded_lora__seed_123,gpt_oss_arc_backloaded_lora__seed_123,gpt_oss_arc_backloaded_lora__seed_123,xwcapimr,sebi/routing_curriculum/xwcapimr,http://localhost:8080/sebi/routing_curriculum/...,2026-04-27T18:55:14Z,finished,routing_curriculum,None,None,174.536592
1,gpt_oss_arc_backloaded_lora__seed_42,gpt_oss_arc_backloaded_lora__seed_42,gpt_oss_arc_backloaded_lora__seed_42,dzc84zc4,sebi/routing_curriculum/dzc84zc4,http://localhost:8080/sebi/routing_curriculum/...,2026-04-27T18:50:35Z,finished,routing_curriculum,None,None,176.667594
2,gpt_oss_arc_backloaded_lora__seed_999,gpt_oss_arc_backloaded_lora__seed_999,gpt_oss_arc_backloaded_lora__seed_999,rptnit2c,sebi/routing_curriculum/rptnit2c,http://localhost:8080/sebi/routing_curriculum/...,2026-04-27T18:59:50Z,finished,routing_curriculum,None,None,173.965239
3,gpt_oss_arc_fixed_k_1_lora__seed_123,gpt_oss_arc_fixed_k_1_lora__seed_123,gpt_oss_arc_fixed_k_1_lora__seed_123,8t33ycuo,sebi/routing_curriculum/8t33ycuo,http://localhost:8080/sebi/routing_curriculum/...,2026-04-27T18:13:20Z,finished,routing_curriculum,None,None,160.782717
4,gpt_oss_arc_fixed_k_1_lora__seed_42,gpt_oss_arc_fixed_k_1_lora__seed_42,gpt_oss_arc_fixed_k_1_lora__seed_42,c62u5reo,sebi/routing_curriculum/c62u5reo,http://localhost:8080/sebi/routing_curriculum/...,2026-04-27T18:08:11Z,finished,routing_curriculum,None,None,160.006706
...,...,...,...,...,...,...,...,...,...,...,...,...
283,qwen_sciq_linear_mid_start_lora__seed_42,qwen_sciq_linear_mid_start_lora__seed_42,qwen_sciq_linear_mid_start_lora__seed_42,ttgaoouq,sebi/routing_curriculum/ttgaoouq,http://localhost:8080/sebi/routing_curriculum/...,2026-04-25T09:24:20Z,finished,routing_curriculum,None,None,625.495132
284,qwen_sciq_linear_mid_start_lora__seed_999,qwen_sciq_linear_mid_start_lora__seed_999,qwen_sciq_linear_mid_start_lora__seed_999,t810zfg2,sebi/routing_curriculum/t810zfg2,http://localhost:8080/sebi/routing_curriculum/...,2026-04-25T09:49:20Z,finished,routing_curriculum,None,None,639.469865
285,qwen_sciq_warmup_lora__seed_123,qwen_sciq_warmup_lora__seed_123,qwen_sciq_warmup_lora__seed_123,uizcnfc3,sebi/routing_curriculum/uizcnfc3,http://localhost:8080/sebi/routing_curriculum/...,2026-04-25T10:14:44Z,finished,routing_curriculum,None,None,650.838188
286,qwen_sciq_warmup_lora__seed_42,qwen_sciq_warmup_lora__seed_42,qwen_sciq_warmup_lora__seed_42,g98ka3fi,sebi/routing_curriculum/g98ka3fi,http://localhost:8080/sebi/routing_curriculum/...,2026-04-25T10:02:01Z,finished,routing_curriculum,None,None,639.436350


In [9]:
duplicate_wandb_runs

,run_name,num_wandb_matches


## 3. Join Eval Metrics with W&B Runtimes


In [10]:
comparison_table = eval_df.merge(
    wandb_df,
    on="run_name",
    how="left",
    validate="one_to_one",
)

print(f"Rows missing matched runtime: {comparison_table['training_runtime_seconds'].isna().sum()}")
comparison_table.sort_values(["benchmark", "model_family", "curriculum"]).reset_index(drop=True)

Rows missing matched runtime: 0


,benchmark,model_family,curriculum,run_name,train_run_name_x,seed,metric_name,metric_value,num_examples,checkpoint_path,...,wandb_run_name,wandb_run_id,wandb_run_path,wandb_run_url,created_at,state,wandb_project,dataset_id,output_dir,training_runtime_seconds
0,arc,gpt_oss,backloaded,gpt_oss_arc_backloaded_lora__seed_42,gpt_oss_arc_backloaded_lora__seed_42,42,accuracy,0.876280,1172,ckpt/gpt_oss_arc_backloaded_lora__seed_42/fina...,...,gpt_oss_arc_backloaded_lora__seed_42,dzc84zc4,sebi/routing_curriculum/dzc84zc4,http://localhost:8080/sebi/routing_curriculum/...,2026-04-27T18:50:35Z,finished,routing_curriculum,None,None,176.667594
1,arc,gpt_oss,backloaded,gpt_oss_arc_backloaded_lora__seed_123,gpt_oss_arc_backloaded_lora__seed_123,123,accuracy,0.869454,1172,ckpt/gpt_oss_arc_backloaded_lora__seed_123/fin...,...,gpt_oss_arc_backloaded_lora__seed_123,xwcapimr,sebi/routing_curriculum/xwcapimr,http://localhost:8080/sebi/routing_curriculum/...,2026-04-27T18:55:14Z,finished,routing_curriculum,None,None,174.536592
2,arc,gpt_oss,backloaded,gpt_oss_arc_backloaded_lora__seed_999,gpt_oss_arc_backloaded_lora__seed_999,999,accuracy,0.865188,1172,ckpt/gpt_oss_arc_backloaded_lora__seed_999/fin...,...,gpt_oss_arc_backloaded_lora__seed_999,rptnit2c,sebi/routing_curriculum/rptnit2c,http://localhost:8080/sebi/routing_curriculum/...,2026-04-27T18:59:50Z,finished,routing_curriculum,None,None,173.965239
3,arc,gpt_oss,fixed_k_1,gpt_oss_arc_fixed_k_1_lora__seed_42,gpt_oss_arc_fixed_k_1_lora__seed_42,42,accuracy,0.747440,1172,ckpt/gpt_oss_arc_fixed_k_1_lora__seed_42/final...,...,gpt_oss_arc_fixed_k_1_lora__seed_42,c62u5reo,sebi/routing_curriculum/c62u5reo,http://localhost:8080/sebi/routing_curriculum/...,2026-04-27T18:08:11Z,finished,routing_curriculum,None,None,160.006706
4,arc,gpt_oss,fixed_k_1,gpt_oss_arc_fixed_k_1_lora__seed_123,gpt_oss_arc_fixed_k_1_lora__seed_123,123,accuracy,0.763652,1172,ckpt/gpt_oss_arc_fixed_k_1_lora__seed_123/fina...,...,gpt_oss_arc_fixed_k_1_lora__seed_123,8t33ycuo,sebi/routing_curriculum/8t33ycuo,http://localhost:8080/sebi/routing_curriculum/...,2026-04-27T18:13:20Z,finished,routing_curriculum,None,None,160.782717
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,sciq,qwen,linear_mid_start,qwen_sciq_linear_mid_start_lora__seed_123,qwen_sciq_linear_mid_start_lora__seed_123,123,accuracy,0.927000,1000,ckpt/qwen_sciq_linear_mid_start_lora__seed_123...,...,qwen_sciq_linear_mid_start_lora__seed_123,e9zr6ord,sebi/routing_curriculum/e9zr6ord,http://localhost:8080/sebi/routing_curriculum/...,2026-04-25T09:36:49Z,finished,routing_curriculum,None,None,627.466778
284,sciq,qwen,linear_mid_start,qwen_sciq_linear_mid_start_lora__seed_999,qwen_sciq_linear_mid_start_lora__seed_999,999,accuracy,0.923000,1000,ckpt/qwen_sciq_linear_mid_start_lora__seed_999...,...,qwen_sciq_linear_mid_start_lora__seed_999,t810zfg2,sebi/routing_curriculum/t810zfg2,http://localhost:8080/sebi/routing_curriculum/...,2026-04-25T09:49:20Z,finished,routing_curriculum,None,None,639.469865
285,sciq,qwen,warmup,qwen_sciq_warmup_lora__seed_42,qwen_sciq_warmup_lora__seed_42,42,accuracy,0.922000,1000,ckpt/qwen_sciq_warmup_lora__seed_42/final_adapter,...,qwen_sciq_warmup_lora__seed_42,g98ka3fi,sebi/routing_curriculum/g98ka3fi,http://localhost:8080/sebi/routing_curriculum/...,2026-04-25T10:02:01Z,finished,routing_curriculum,None,None,639.436350
286,sciq,qwen,warmup,qwen_sciq_warmup_lora__seed_123,qwen_sciq_warmup_lora__seed_123,123,accuracy,0.929000,1000,ckpt/qwen_sciq_warmup_lora__seed_123/final_ada...,...,qwen_sciq_warmup_lora__seed_123,uizcnfc3,sebi/routing_curriculum/uizcnfc3,http://localhost:8080/sebi/routing_curriculum/...,2026-04-25T10:14:44Z,finished,routing_curriculum,None,None,650.838188


## 4. Compute Baseline Deltas

Baseline is the `fixed_k_max` run within each `(benchmark, model_family)` group.


In [11]:
def absolute_delta(value, baseline):
    if pd.isna(value) or pd.isna(baseline):
        return pd.NA
    return value - baseline


def pct_delta(value, baseline):
    if pd.isna(value) or pd.isna(baseline) or baseline == 0:
        return pd.NA
    return (value - baseline) / baseline * 100.0


BASELINE_CURRICULUM = "fixed_k_max"

grouped_comparison_table = (
    comparison_table.groupby(["benchmark", "model_family", "curriculum"], dropna=False)
    .agg(
        metric_name=("metric_name", "first"),
        metric_value_mean=("metric_value", "mean"),
        metric_value_std=(
            "metric_value",
            lambda values: pd.to_numeric(values, errors="coerce").std(ddof=0),
        ),
        training_runtime_seconds_mean=("training_runtime_seconds", "mean"),
        training_runtime_seconds_std=(
            "training_runtime_seconds",
            lambda values: pd.to_numeric(values, errors="coerce").std(ddof=0),
        ),
        num_runs=("run_name", "nunique"),
    )
    .reset_index()
)

baseline_table = (
    grouped_comparison_table.loc[
        grouped_comparison_table["curriculum"] == BASELINE_CURRICULUM,
        ["benchmark", "model_family", "metric_value_mean", "training_runtime_seconds_mean"],
    ]
    .rename(
        columns={
            "metric_value_mean": "baseline_metric_value_mean",
            "training_runtime_seconds_mean": "baseline_training_runtime_seconds_mean",
        }
    )
)

baseline_table


,benchmark,model_family,baseline_metric_value_mean,baseline_training_runtime_seconds_mean
2,arc,gpt_oss,0.882253,202.544924
10,arc,lfm2,0.842435,122.924492
18,arc,olmoe,0.676906,109.318323
26,arc,qwen,0.781854,196.169920
34,gsm8k,gpt_oss,0.716452,652.221471
42,gsm8k,lfm2,0.612585,318.107677
50,gsm8k,olmoe,0.368461,281.207475
58,gsm8k,qwen,0.409401,565.837574
66,sciq,gpt_oss,0.954333,620.000278
74,sciq,lfm2,0.952333,410.937269


In [12]:
grouped_comparison_table = grouped_comparison_table.merge(
    baseline_table,
    on=["benchmark", "model_family"],
    how="left",
    validate="many_to_one",
)

grouped_comparison_table["metric_delta"] = grouped_comparison_table.apply(
    lambda row: absolute_delta(row["metric_value_mean"], row["baseline_metric_value_mean"]),
    axis=1,
)
grouped_comparison_table["runtime_delta_pct"] = grouped_comparison_table.apply(
    lambda row: pct_delta(
        row["training_runtime_seconds_mean"],
        row["baseline_training_runtime_seconds_mean"],
    ),
    axis=1,
)
grouped_comparison_table["is_baseline"] = grouped_comparison_table["curriculum"].eq(
    BASELINE_CURRICULUM
)

grouped_comparison_table = grouped_comparison_table[
    [
        "benchmark",
        "model_family",
        "curriculum",
        "num_runs",
        "metric_name",
        "metric_value_mean",
        "metric_value_std",
        "baseline_metric_value_mean",
        "metric_delta",
        "training_runtime_seconds_mean",
        "training_runtime_seconds_std",
        "baseline_training_runtime_seconds_mean",
        "runtime_delta_pct",
        "is_baseline",
    ]
].sort_values(["benchmark", "model_family", "curriculum"]).reset_index(drop=True)

grouped_comparison_table


,benchmark,model_family,curriculum,num_runs,metric_name,metric_value_mean,metric_value_std,baseline_metric_value_mean,metric_delta,training_runtime_seconds_mean,training_runtime_seconds_std,baseline_training_runtime_seconds_mean,runtime_delta_pct,is_baseline
0,arc,gpt_oss,backloaded,3,accuracy,0.870307,0.004568,0.882253,-0.011945,175.056475,1.162867,202.544924,-13.571532,False
1,arc,gpt_oss,fixed_k_1,3,accuracy,0.763936,0.013587,0.882253,-0.118316,160.879091,0.754727,202.544924,-20.571157,False
2,arc,gpt_oss,fixed_k_max,3,accuracy,0.882253,0.003686,0.882253,0.000000,202.544924,0.246081,202.544924,0.000000,True
3,arc,gpt_oss,frontloaded,3,accuracy,0.885950,0.001450,0.882253,0.003697,190.089046,0.776533,202.544924,-6.149687,False
4,arc,gpt_oss,jump_warmup,3,accuracy,0.889647,0.007768,0.882253,0.007395,196.995836,1.202995,202.544924,-2.739683,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,sciq,qwen,frontloaded,3,accuracy,0.925667,0.001247,0.932333,-0.006667,629.650070,4.111402,646.388119,-2.589474,False
92,sciq,qwen,jump_warmup,3,accuracy,0.931667,0.001247,0.932333,-0.000667,635.435605,0.824960,646.388119,-1.694418,False
93,sciq,qwen,linear_k_1_to_topk,3,accuracy,0.926000,0.002160,0.932333,-0.006333,617.295751,6.986648,646.388119,-4.500759,False
94,sciq,qwen,linear_mid_start,3,accuracy,0.926000,0.002160,0.932333,-0.006333,630.810592,6.175711,646.388119,-2.409934,False


In [13]:
display_table = grouped_comparison_table.copy()
for column in [
    "metric_value_mean",
    "metric_value_std",
    "baseline_metric_value_mean",
    "metric_delta",
]:
    display_table[column] = pd.to_numeric(display_table[column], errors="coerce").round(3)

for column in [
    "training_runtime_seconds_mean",
    "training_runtime_seconds_std",
    "baseline_training_runtime_seconds_mean",
    "runtime_delta_pct",
]:
    display_table[column] = pd.to_numeric(display_table[column], errors="coerce").round(2)

display_table


,benchmark,model_family,curriculum,num_runs,metric_name,metric_value_mean,metric_value_std,baseline_metric_value_mean,metric_delta,training_runtime_seconds_mean,training_runtime_seconds_std,baseline_training_runtime_seconds_mean,runtime_delta_pct,is_baseline
0,arc,gpt_oss,backloaded,3,accuracy,0.870,0.005,0.882,-0.012,175.06,1.16,202.54,-13.57,False
1,arc,gpt_oss,fixed_k_1,3,accuracy,0.764,0.014,0.882,-0.118,160.88,0.75,202.54,-20.57,False
2,arc,gpt_oss,fixed_k_max,3,accuracy,0.882,0.004,0.882,0.000,202.54,0.25,202.54,0.00,True
3,arc,gpt_oss,frontloaded,3,accuracy,0.886,0.001,0.882,0.004,190.09,0.78,202.54,-6.15,False
4,arc,gpt_oss,jump_warmup,3,accuracy,0.890,0.008,0.882,0.007,197.00,1.20,202.54,-2.74,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,sciq,qwen,frontloaded,3,accuracy,0.926,0.001,0.932,-0.007,629.65,4.11,646.39,-2.59,False
92,sciq,qwen,jump_warmup,3,accuracy,0.932,0.001,0.932,-0.001,635.44,0.82,646.39,-1.69,False
93,sciq,qwen,linear_k_1_to_topk,3,accuracy,0.926,0.002,0.932,-0.006,617.30,6.99,646.39,-4.50,False
94,sciq,qwen,linear_mid_start,3,accuracy,0.926,0.002,0.932,-0.006,630.81,6.18,646.39,-2.41,False


In [14]:
reduced_table = display_table[[
    "benchmark",
    "model_family",
    "curriculum",
    "metric_value_mean",
    "metric_value_std",
    "metric_delta",
    "training_runtime_seconds_mean",
    "training_runtime_seconds_std",
    "runtime_delta_pct",
]].copy()


def format_mean_std(mean_value, std_value, precision, suffix=""):
    if pd.isna(mean_value):
        return pd.NA

    std_value = 0 if pd.isna(std_value) else std_value
    return f"{mean_value:.{precision}f}{suffix} ± {std_value:.{precision}f}{suffix}"


reduced_table["metric_value"] = reduced_table.apply(
    lambda row: (
        f"{format_mean_std(row['metric_value_mean'], row['metric_value_std'], 3)} ({row['metric_delta']:+.3f})"
        if (row["curriculum"] != BASELINE_CURRICULUM and not pd.isna(row["metric_delta"]))
        else format_mean_std(row["metric_value_mean"], row["metric_value_std"], 3)
    ),
    axis=1,
)

reduced_table["training_runtime_seconds"] = reduced_table.apply(
    lambda row: (
        f"{format_mean_std(row['training_runtime_seconds_mean'], row['training_runtime_seconds_std'], 2, 's')} ({row['runtime_delta_pct']:+.2f}%)"
        if (row["curriculum"] != BASELINE_CURRICULUM and not pd.isna(row["runtime_delta_pct"]))
        else format_mean_std(row["training_runtime_seconds_mean"], row["training_runtime_seconds_std"], 2, "s")
    ),
    axis=1,
)

reduced_table = reduced_table.drop(columns=[
    "metric_value_mean",
    "metric_value_std",
    "metric_delta",
    "training_runtime_seconds_mean",
    "training_runtime_seconds_std",
    "runtime_delta_pct",
])

curriculum_rank = {name: index for index, name in enumerate(CURRICULUM_DISPLAY_ORDER)}
reduced_table["curriculum_sort_key"] = reduced_table["curriculum"].map(
    lambda value: curriculum_rank.get(value, len(curriculum_rank))
)
reduced_table = reduced_table.sort_values(
    ["benchmark", "model_family", "curriculum_sort_key", "curriculum"]
).drop(columns=["curriculum_sort_key"]).reset_index(drop=True)
reduced_table


,benchmark,model_family,curriculum,metric_value,training_runtime_seconds
0,arc,gpt_oss,fixed_k_max,0.882 ± 0.004,202.54s ± 0.25s
1,arc,gpt_oss,fixed_k_1,0.764 ± 0.014 (-0.118),160.88s ± 0.75s (-20.57%)
2,arc,gpt_oss,linear_k_1_to_topk,0.882 ± 0.005 (+0.000),182.04s ± 0.41s (-10.12%)
3,arc,gpt_oss,frontloaded,0.886 ± 0.001 (+0.004),190.09s ± 0.78s (-6.15%)
4,arc,gpt_oss,backloaded,0.870 ± 0.005 (-0.012),175.06s ± 1.16s (-13.57%)
...,...,...,...,...,...
91,sciq,qwen,frontloaded,0.926 ± 0.001 (-0.007),629.65s ± 4.11s (-2.59%)
92,sciq,qwen,backloaded,0.931 ± 0.002 (-0.002),607.25s ± 1.03s (-6.06%)
93,sciq,qwen,warmup,0.924 ± 0.003 (-0.008),642.46s ± 6.00s (-0.61%)
94,sciq,qwen,jump_warmup,0.932 ± 0.001 (-0.001),635.44s ± 0.82s (-1.69%)


In [15]:
curriculum_rank = {name: index for index, name in enumerate(CURRICULUM_DISPLAY_ORDER)}
available_curricula = sorted(
    set(reduced_table["curriculum"].dropna()),
    key=lambda value: (curriculum_rank.get(value, len(curriculum_rank)), value),
)

CURRICULUM_LABELS = {
    "fixed_k_max": "Fixed K Default",
}


def curriculum_label(curriculum):
    return CURRICULUM_LABELS.get(curriculum, curriculum.replace("_", " ").title())


metric_table = reduced_table.pivot(
    index=["benchmark", "model_family"],
    columns="curriculum",
    values="metric_value",
)
metric_table = metric_table.reindex(columns=available_curricula)
metric_table.columns = [curriculum_label(curriculum) for curriculum in metric_table.columns]

runtime_table = reduced_table.pivot(
    index=["benchmark", "model_family"],
    columns="curriculum",
    values="training_runtime_seconds",
)
runtime_table = runtime_table.reindex(columns=available_curricula)
runtime_table.columns = [curriculum_label(curriculum) for curriculum in runtime_table.columns]

delta_table = grouped_comparison_table.pivot(
    index=["benchmark", "model_family"],
    columns="curriculum",
    values=["metric_delta", "runtime_delta_pct"],
)
delta_table = delta_table.reindex(
    columns=pd.MultiIndex.from_product(
        [["metric_delta", "runtime_delta_pct"], available_curricula]
    )
)
delta_table = delta_table.swaplevel(0, 1, axis=1)

metric_delta_table = delta_table.xs("metric_delta", axis=1, level=1)
metric_delta_table.columns = [curriculum_label(curriculum) for curriculum in metric_delta_table.columns]

runtime_delta_table = delta_table.xs("runtime_delta_pct", axis=1, level=1)
runtime_delta_table.columns = [curriculum_label(curriculum) for curriculum in runtime_delta_table.columns]


def highlight_metric_cells(row):
    styles = pd.Series("", index=row.index)
    delta_row = metric_delta_table.loc[row.name]

    for column in row.index:
        if column == curriculum_label(BASELINE_CURRICULUM):
            continue

        delta_value = delta_row.get(column)
        if pd.isna(delta_value) or delta_value == 0:
            continue

        color = "#56ad34" if delta_value > 0 else "#d65f5f"
        styles[column] = f"background-color: {color}; font-weight: 600"

    return styles


def highlight_runtime_cells(row):
    styles = pd.Series("", index=row.index)
    delta_row = runtime_delta_table.loc[row.name]

    for column in row.index:
        if column == curriculum_label(BASELINE_CURRICULUM):
            continue

        delta_value = delta_row.get(column)
        if pd.isna(delta_value) or delta_value == 0:
            continue

        color = "#578aba" if delta_value < 0 else "#e6c84f"
        styles[column] = f"background-color: {color}; font-weight: 600"

    return styles


(
    metric_table.style.set_caption("Grouped eval metric deltas (mean ± std)")
    .apply(highlight_metric_cells, axis=1)
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "bottom"), ("font-size", "0.95em")]},
        {"selector": "th.col_heading", "props": [("text-align", "center"), ("border-bottom", "1px solid #000")]},
        {"selector": "th.row_heading", "props": [("text-align", "left")]},
    ])
    .set_properties(**{"text-align": "center", "white-space": "nowrap"})
)


In [16]:
(
    runtime_table.style.set_caption("Grouped eval runtime deltas (mean ± std)")
    .apply(highlight_runtime_cells, axis=1)
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "bottom"), ("font-size", "0.95em")]},
        {"selector": "th.col_heading", "props": [("text-align", "center"), ("border-bottom", "1px solid #000")]},
        {"selector": "th.row_heading", "props": [("text-align", "left")]},
    ])
    .set_properties(**{"text-align": "center", "white-space": "nowrap"})
)
